# American Airlines Support Agent — Exploratory Analysis & Intent Clustering

This notebook covers:
1. Brand Selection & Volume Sanity Checks
2. AmericanAir Thread Ingestion & Inspection (20 Random Samples)
3. Traffic Volume & Conversation Turn Distributions
4. Bottom-up Intent Clustering (KMeans k=6..12 silhouette sweep)
5. Historical Retrieval Grounding Checks

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

DATA_PATH = "data/processed/aa_threads.parquet"
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} processed AmericanAir threads.")
df.head(3)

## 1. Thread Statistics & Sanity Checks

In [2]:
print("=== AMERICAN AIRLINES THREAD METRICS ===")
print(f"Total Threads       : {len(df):,}")
print(f"Average Turns/Thread: {df['turns_count'].mean():.2f}")
print(f"Max Turns in Thread : {df['turns_count'].max()}")
print(f"Boilerplate Replies : {df['is_boilerplate'].sum():,} ({df['is_boilerplate'].mean()*100:.2f}%)")
print(f"Avg Cust Msg Words  : {df['customer_message'].str.split().str.len().mean():.1f}")
print(f"Avg AA Reply Words  : {df['aa_reply'].str.split().str.len().mean():.1f}")

## 2. Random Sample of 20 Raw Threads (Gut Check Inspection)
Inspecting real customer inquiries and the corresponding first response from @AmericanAir.

In [3]:
sample_20 = df.sample(n=20, random_state=42).reset_index(drop=True)
for i, row in sample_20.iterrows():
    print(f"\n[{i+1:02d}] Thread ID: {row['thread_id']} (Turns: {row['turns_count']}, Boilerplate: {row['is_boilerplate']})")
    print(f"  Customer: {row['customer_message']}")
    print(f"  AA Reply: {row['aa_reply']}")

## 3. Intent Clustering Analysis (k=6 through k=12)
Evaluating unsupervised KMeans clustering silhouette scores on customer opening turns.

In [4]:
from src.intents import IntentClassifier, INTENT_TAXONOMY, evaluate_clustering

# Evaluate clustering sweep
cluster_res = evaluate_clustering(df, sample_size=3000, seed=42)
for k, met in cluster_res.items():
    print(f"k={k:2d} | Silhouette: {met['silhouette']:.4f} | Cluster Sizes: {met['cluster_distribution']}")

## 4. Grounding Retrieval Verification
Testing historical grounding retrieval on sample queries across different intents.

In [5]:
from src.retrieval import RetrievalIndex

retriever = RetrievalIndex()
test_inquiries = [
    ("Delayed in Philadelphia flight AA 1083 missed my connect to London", "flight_delay_cancellation"),
    ("Bag never showed up at baggage carousel in Phoenix", "baggage_issue"),
    ("Can I rebook for an earlier flight this afternoon?", "rebooking_change_request"),
    ("Requesting a ticket refund for cancelled flight yesterday", "refund_compensation_request"),
    ("My AAdvantage account is not reflecting miles from last week", "aadvantage_miles_issue")
]

for query, intent in test_inquiries:
    res = retriever.retrieve(query, predicted_intent=intent, top_k=2)
    print(f"\n[Query] {query}")
    print(f"  Predicted Intent : {intent} | Best Similarity: {res['best_similarity']:.3f}")
    for m in res["matches"]:
        print(f"    Precedent ({m['raw_similarity']:.3f}): \"{m['aa_reply'][:90]}...\"")